# Team 13 · Monthly passenger demand
Deliverable 3 explores observed directional airport pairs by departure month. The response is passengers (`OD_PAX`). Forecasts are issued before the first forecast month for a supplied set of routes. The final horizon and coverage require team confirmation. Calendar and route identity can be specified for any horizon.

Run all cells locally with the approved CSVs under `data/raw/`. Keep the source notebook cleared when committing. The audit, figures and limitations are written to ignored `data/interim/team-13-d3/`. The last cell exposes a feature frame, response and dates for chronological splitting in D4.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.git').exists())
raw = root / 'data/raw'
out = root / 'data/interim/team-13-d3'
out.mkdir(parents=True, exist_ok=True)

def read_source(name):
    matches = [p for p in raw.glob(f'{name}*.csv') if p.stem in (name, f'{name} - {name}')]
    if len(matches) != 1:
        raise ValueError(f'Expected one local copy of {name}')
    return pd.read_csv(matches[0])

demand = read_source('AA_PAX_FORECASTING')
airports = read_source('AA_AIRPORT_INFO')
audit = []
def record(title, value):
    audit.append(f'{title}\n{value}\n')
    print(title, value, sep='\n')

for name, frame in [('Demand', demand), ('Airport reference', airports)]:
    record(name + ' dimensions', frame.shape)
    record(name + ' schema', pd.DataFrame({'type': frame.dtypes, 'missing': frame.isna().sum()}))


In [ ]:
route = ['ORG_AIRPRT', 'DES_AIRPRT']
key = ['month', *route]
panel = demand.copy()
for field in route:
    panel[field] = panel[field].astype('string').str.strip().replace('', pd.NA)
panel['month'] = pd.to_datetime(dict(year=panel.DEP_YEAR, month=panel.DEP_MONTH, day=1), errors='coerce')
panel['OD_PAX'] = pd.to_numeric(panel.OD_PAX, errors='coerce')
invalid = panel[key].isna().any(axis=1) | ~np.isfinite(panel.OD_PAX) | panel.OD_PAX.lt(0)
record('Rows excluded for missing keys or invalid passenger counts', int(invalid.sum()))
panel = panel.loc[~invalid].sort_values(key).reset_index(drop=True)
if panel.empty or panel.duplicated(key).any():
    raise ValueError('Resolve the route-month grain with the team before building features.')
record('Observed months', (panel.month.min(), panel.month.max()))
coverage = panel.groupby(route).month.agg(['min', 'max', 'size'])
span = (coverage['max'].dt.year - coverage['min'].dt.year)*12 + coverage['max'].dt.month - coverage['min'].dt.month + 1
record('Routes / absent months within observed route lifespans', (len(coverage), int((span-coverage['size']).sum())))
record('Observed months per route', coverage['size'].describe())
record('Passenger distribution', panel.OD_PAX.describe(percentiles=[.25,.5,.75,.95,.99]))
q1, q3 = panel.OD_PAX.quantile([.25,.75])
record('Rows above upper IQR fence (retained for review)', int(panel.OD_PAX.gt(q3 + 1.5*(q3-q1)).sum()))
record('Missing demand fields by year', demand.groupby('DEP_YEAR').agg(lambda s: s.isna().mean()).to_string())


In [ ]:
lookup = airports[['AIRPRT_CD', 'CNTRY_CD']].copy()
lookup['AIRPRT_CD'] = lookup.AIRPRT_CD.astype('string').str.strip().replace('', pd.NA)
lookup = lookup.dropna(subset=['AIRPRT_CD'])
if lookup.AIRPRT_CD.duplicated().any():
    raise ValueError('Airport key is ambiguous; reconcile lookup records first.')
for airport in route:
    matched = panel[[airport]].merge(lookup, left_on=airport, right_on='AIRPRT_CD',
                                     how='left', validate='many_to_one', indicator=True)
    assert len(matched) == len(panel)
    record(airport + ' lookup coverage', matched['_merge'].eq('both').mean())
    record(airport + ' missing country fraction', matched.CNTRY_CD.isna().mean())

monthly = panel.groupby('month').OD_PAX.agg(['sum','size']).asfreq('MS')
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
monthly['sum'].plot(ax=axes[0], ylabel='Observed passengers')
monthly['size'].plot(ax=axes[1], ylabel='Observed routes', xlabel='Departure month')
fig.tight_layout()
fig.savefig(out / 'monthly-demand-and-coverage.png')
season = panel.groupby(panel.month.dt.month).OD_PAX.agg(['mean','median'])
ax = season.plot(marker='o', xlabel='Departure month (1–12)', ylabel='Passengers per observed route-month')
ax.figure.tight_layout()
ax.figure.savefig(out / 'seasonal-demand.png')
record('Observed calendar-month distribution', season)
record('Coverage interpretation', f"Monthly observations range from {int(monthly['size'].min())} to {int(monthly['size'].max())} among represented months. Changes in totals can reflect coverage. Absent records have unknown demand.")
record('Distribution interpretation', f"The median is {panel.OD_PAX.median():.1f} passengers versus a mean of {panel.OD_PAX.mean():.1f}. Inspect the upper tail before choosing an error metric. Calendar-month differences combine route mix and possible seasonality.")
(out / 'audit.txt').write_text('\n'.join(audit), encoding='utf-8')


### Feature decision for D4

Use a pooled gradient-boosted tree regressor as the candidate family. Each route contributes observations to a shared model. Splits can represent nonlinear calendar effects and different seasonal responses by airport pair. The audit quantifies route coverage and demand dispersion for judging that choice. D4 must test predictive utility on later months.

| Field | Decision and reason |
|---|---|
| Origin and destination | Separate nominal features preserve direction and allow information sharing across routes. Fit a categorical encoder on training dates only, with unknown-category handling. |
| Departure year | Known at issue time. Captures historical changes within the training range. Trees cannot extrapolate a trend beyond that range. |
| Calendar month sine and cosine | Known at issue time. December and January are adjacent on this scale. |
| Passenger count | Response only. Missing route-months are left unobserved. |
| Same-month revenue | Realized after the forecast cutoff and jointly determined with passenger volume. |
| Export counter and warehouse timestamp | Ingestion metadata with no established forecasting meaning. |
| Airport lookup attributes | Used for coverage diagnostics. Historical availability of the reference snapshot is unverified. |
| Passenger lags and rolling means | Deferred until the issue date, publication lag and operational forecast horizon are confirmed. |

In [ ]:
def build_features(rows):
    """Create predictors from known directional markets and their departure months."""
    features = rows[route].astype('string').copy()
    dates = rows['month'].dt
    angle = 2 * np.pi * (dates.month - 1) / 12
    features['departure_year'] = dates.year
    features['month_sin'] = np.sin(angle)
    features['month_cos'] = np.cos(angle)
    return features

X = build_features(panel[key])
y = panel['OD_PAX'].copy()
months = panel['month'].copy()
assert X.index.equals(y.index) and X.index.equals(months.index)
assert X.notna().all().all() and months.is_monotonic_increasing
